# 📖 Notebook 8: Service Mesh — Istio for Traffic Management and Security

Welcome to the service mesh lab. In plain language, a **service mesh** gives your apps a smart helper layer for service-to-service communication. Instead of teaching every app how to do retries, encryption, traffic shaping, and detailed observability, the mesh handles much of that work beside your app.

In this lab, we will use Istio with the sample microservices in the `k8s-lab` namespace:
- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

A simple way to think about it is: **your app focuses on business logic, and the mesh focuses on how services talk safely and reliably.**


## Learning Objectives

By the end of this notebook, you will be able to:

- explain what a service mesh is and why teams use one
- describe Istio sidecars in beginner-friendly terms
- enable mTLS for service-to-service encryption
- split traffic between service versions for canary releases
- add retries and timeouts without changing application code
- view service relationships with Kiali


## 🛠️ Setup

Before starting:

1. Make sure your Kubernetes cluster is running.
2. Make sure the `k8s-lab` namespace and sample services are already deployed.
3. Istio demo installs several extra components, so a cluster with **6 GB of memory or more** is strongly recommended.
4. Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

This notebook uses shell commands so you can see the mesh setup step by step.


In [ ]:
!kubectl cluster-info
!kubectl get nodes -o wide
!kubectl get deployments -n k8s-lab


## What is a Service Mesh?

A service mesh is a **dedicated infrastructure layer for service-to-service communication**. That sentence sounds big, so let us shrink it:

- your services still send HTTP requests like normal
- small helper proxies sit next to the services
- those proxies handle networking features for you

This matters because distributed systems get complicated quickly. Even a small app can need encryption, retries, routing rules, and visibility into failures. A mesh centralizes those concerns.

### 🧪 Practical Exercise
Look at the three services in `k8s-lab` and imagine adding retries and encryption to every service by hand. Which approach feels easier to maintain: changing all apps or adding shared traffic rules in one place?


## 🔀 Pod A → Sidecar Proxy → Sidecar Proxy → Pod B

```text
+-----------+      +------------------+      +------------------+      +-----------+
|   Pod A   | ---> | Sidecar Proxy A  | ---> | Sidecar Proxy B  | ---> |   Pod B   |
| app code  |      | handles traffic  |      | handles traffic  |      | app code  |
+-----------+      +------------------+      +------------------+      +-----------+
```

The key idea is that the app does not need to know every networking trick. The sidecar proxies take care of many cross-cutting concerns.


## Why use a mesh?

A service mesh can provide these features **without changing your app code**:

- **mTLS** for encrypted service-to-service traffic
- **retries** when a request fails briefly
- **timeouts** so requests do not hang forever
- **traffic splitting** for canary releases
- **observability** so you can see service relationships and traffic flow

### 🧪 Practical Exercise
Pick one feature from the list and explain what problem it solves. For example, why might a timeout be better than waiting forever?


In [ ]:
!kubectl get svc -n k8s-lab
!kubectl get pods -n k8s-lab


## 1) Install Istio

We will install the Istio demo profile. It includes enough features for a learning lab.

⚠️ Security Warning: the quick-start command below pipes a downloaded script into `sh` because that matches the common Istio lab flow. In a stricter environment, download the script first, inspect it, and only then run it.

### 🧪 Practical Exercise
Before running the install, predict what new namespace will appear and what kind of pods you expect to see there after Istio is installed.


In [ ]:
!curl -L https://istio.io/downloadIstio | sh -
!export PATH="$PWD/$(find . -maxdepth 1 -type d -name 'istio-*' | head -n 1)/bin:$PATH" && istioctl install --set profile=demo -y
!kubectl get pods -n istio-system


## 2) Enable sidecar injection

Istio adds sidecar proxies when a namespace is labeled for injection. After labeling the namespace, we restart the workloads so new pods are created with the sidecars attached.

### 🧪 Practical Exercise
After the restart, inspect the pod READY column. If a pod shows `2/2`, what do you think the two containers are?


In [ ]:
!kubectl label namespace k8s-lab istio-injection=enabled --overwrite
!kubectl rollout restart deployment --all -n k8s-lab
!kubectl get pods -n k8s-lab


## 3) Enforce mTLS

**mTLS** means **mutual TLS**. In simple terms, both sides of a connection prove who they are, and the traffic is encrypted. This helps protect traffic moving between services inside the cluster.

We will use a `PeerAuthentication` resource in `STRICT` mode so sidecars require mTLS.

### 🧪 Practical Exercise
Read the YAML and say what the word `STRICT` suggests. Is the mesh allowing plain text traffic, preferring encryption, or requiring encryption?


In [ ]:
!printf '%s\n' 'apiVersion: security.istio.io/v1beta1' 'kind: PeerAuthentication' 'metadata:' '  name: default' '  namespace: k8s-lab' 'spec:' '  mtls:' '    mode: STRICT' > manifests/istio-peer-auth.yaml
!cat manifests/istio-peer-auth.yaml
!kubectl apply -f manifests/istio-peer-auth.yaml
!kubectl get peerauthentication -n k8s-lab


## 4) Verify mTLS

Istio has helper commands that can inspect how traffic is protected. This is useful because encryption is hard to prove just by looking at your application code.

### 🧪 Practical Exercise
Run the check below and look for evidence that the connection is using Istio-managed TLS instead of plain text traffic.


In [ ]:
!export PATH="$PWD/$(find . -maxdepth 1 -type d -name 'istio-*' | head -n 1)/bin:$PATH" && istioctl authn tls-check deployment/api-gateway.k8s-lab


## 5) Traffic splitting for a canary release

A canary release sends a small percentage of traffic to a newer version while most users still hit the stable version. This reduces risk.

In this lab, we will:

1. label the existing `user-service` pods as `v1`
2. create a small `user-service-v2` deployment
3. route 90% of traffic to `v1` and 10% to `v2`

The sample FastAPI app returns the same payload, so the easiest place to observe the split is in Kiali later.

### 🧪 Practical Exercise
Before applying the canary, explain why sending only 10% of traffic to a new version is safer than sending 100% immediately.


In [ ]:
!kubectl patch deployment user-service -n k8s-lab --type merge -p '{"spec":{"template":{"metadata":{"labels":{"version":"v1"}}}}}'
!printf '%s\n' 'apiVersion: apps/v1' 'kind: Deployment' 'metadata:' '  name: user-service-v2' '  namespace: k8s-lab' 'spec:' '  replicas: 1' '  selector:' '    matchLabels:' '      app: user-service' '      version: v2' '  template:' '    metadata:' '      labels:' '        app: user-service' '        version: v2' '    spec:' '      containers:' '        - name: user-service' '          image: k8s-lab/user-service:latest' '          ports:' '            - containerPort: 8001' '---' 'apiVersion: networking.istio.io/v1beta1' 'kind: DestinationRule' 'metadata:' '  name: user-service' '  namespace: k8s-lab' 'spec:' '  host: user-service' '  subsets:' '    - name: v1' '      labels:' '        version: v1' '    - name: v2' '      labels:' '        version: v2' '---' 'apiVersion: networking.istio.io/v1beta1' 'kind: VirtualService' 'metadata:' '  name: user-service' '  namespace: k8s-lab' 'spec:' '  hosts:' '    - user-service' '  http:' '    - route:' '        - destination:' '            host: user-service' '            subset: v1' '          weight: 90' '        - destination:' '            host: user-service' '            subset: v2' '          weight: 10' > manifests/istio-canary.yaml
!kubectl apply -f manifests/istio-canary.yaml
!kubectl get deployment user-service user-service-v2 -n k8s-lab
!kubectl get virtualservice,destinationrule -n k8s-lab


## 6) Add retries and timeouts

Traffic management is not only about percentages. You can also tell the mesh how patient to be and how many times to retry before giving up.

Below, we update the `VirtualService` so requests to `user-service` use a 2-second timeout and up to 3 retry attempts for certain transient failures.

### 🧪 Practical Exercise
Why might a retry help with a short network hiccup, but not with a permanent application bug?


In [ ]:
!printf '%s\n' 'apiVersion: networking.istio.io/v1beta1' 'kind: VirtualService' 'metadata:' '  name: user-service' '  namespace: k8s-lab' 'spec:' '  hosts:' '    - user-service' '  http:' '    - timeout: 2s' '      retries:' '        attempts: 3' '        perTryTimeout: 1s' '        retryOn: gateway-error,connect-failure,refused-stream' '      route:' '        - destination:' '            host: user-service' '            subset: v1' '          weight: 90' '        - destination:' '            host: user-service' '            subset: v2' '          weight: 10' > manifests/istio-traffic-policy.yaml
!kubectl apply -f manifests/istio-traffic-policy.yaml
!kubectl describe virtualservice user-service -n k8s-lab


## 7) Observability with Kiali

Kiali is a dashboard that helps you see the mesh as a graph. This is one of the most beginner-friendly ways to understand what is talking to what.

### 🧪 Practical Exercise
After opening Kiali, try to identify the direction of requests between `api-gateway` and `user-service`. The goal is to connect the abstract mesh concepts to a picture.


In [ ]:
!kubectl apply -f https://raw.githubusercontent.com/istio/istio/release-1.24/samples/addons/kiali.yaml
!kubectl wait --for=condition=available deployment/kiali -n istio-system --timeout=180s
!kubectl port-forward svc/kiali -n istio-system 20001:20001 &


## 8) Generate traffic and view the topology

Kiali becomes more interesting when there is real traffic to visualize. We will port-forward the API gateway locally and send several requests through it.

### 🧪 Practical Exercise
Open Kiali at `http://localhost:20001`, go to the graph view for `k8s-lab`, and look for edges between services. Can you find the path from `api-gateway` to `user-service`?


In [ ]:
!kubectl port-forward svc/api-gateway -n k8s-lab 8000:8000 &
!for i in {1..20}; do curl -s http://127.0.0.1:8000/users/1 > /dev/null; done
!kubectl get pods -n k8s-lab


## 🧹 Clean Up

When you finish the lab, remove Istio so the cluster returns to a simpler state.

### 🧪 Practical Exercise
After uninstalling Istio, inspect the `k8s-lab` pods again. What do you expect to happen to the sidecars and the READY counts over time?


In [ ]:
!export PATH="$PWD/$(find . -maxdepth 1 -type d -name 'istio-*' | head -n 1)/bin:$PATH" && istioctl uninstall --purge -y
!kubectl label namespace k8s-lab istio-injection-


## 🎓 What You Learned

Great job. In this notebook, you learned that:

- a service mesh adds a dedicated communication layer between services
- Istio uses sidecar proxies to provide mesh features
- mTLS can encrypt and authenticate service-to-service traffic
- traffic splitting supports safer canary rollouts
- retries and timeouts can be configured in mesh policy instead of app code
- Kiali helps you visualize the service graph and traffic flow

If you can explain why sidecars make networking features reusable across many services, you understand the core value of a service mesh.
